In [ ]:
!pip install datasets
from datasets import load_dataset

# Load the dataset
dataset_dict = load_dataset("cnn_dailymail", "3.0.0")

# Convert the 'train', 'validation', and 'test' splits to pandas DataFrames
train_df = dataset_dict['train'].to_pandas()
val_df = dataset_dict['validation'].to_pandas()
test_df = dataset_dict['test'].to_pandas()

# Select the first 100 instances from each split (optional)
train_df = dataset_dict['train'].select(range(100))
val_df =  dataset_dict['validation'].select(range(10))
test_df =  dataset_dict['test'].select(range(10))

# Print the shapes of the DataFrames
print("Training data shape:", train_df.shape)
print("Validation data shape:", val_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (100, 3)
Validation data shape: (10, 3)
Test data shape: (10, 3)


In [ ]:
!pip install keras tensorflow
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [ ]:
class WordAttention(Layer):
    def __init__(self, **kwargs):
        super(WordAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(WordAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        print("Input shape to WordAttention:", x.shape)

        # Get dynamic shape values
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]  # Get num_tokens dynamically from input shape
        feature_dim = x.shape[-1]  # Use the last dimension as feature_dim

        #Calculate uit
        uit = K.tanh(K.dot(x, self.W) + self.b)
        print("uit shape:", uit.shape)

        #Calculate ait
        ait = K.dot(uit, self.u)
        print("ait shape:", ait.shape)

        #Apply softmax to get attention weights
        ait = tf.nn.softmax(ait, axis=1)
        print("ait shape after softmax:", ait.shape)

        #Perform element-wise multiplication
        weighted_input = x * ait # (batch_size, num_tokens, feature_dim)
        print("weighted_input shape:", weighted_input.shape)

        #Sum weighted inputs to get context vector
        output = K.sum(weighted_input, axis=1) # (batch_size, feature_dim)
        print("output shape:", output.shape)

        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
class SentenceAttention(Layer):
    def __init__(self, **kwargs):
        super(SentenceAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(SentenceAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)  # Ensure input is float32
        print("Input shape to SentenceAttention:", x.shape)
        # Ensure correct reshaping based on your specific requirements
        batch_size = tf.shape(x)[0]
        num_sentences = tf.shape(x)[1]
        feature_dim = tf.shape(x)[2] if len(x.shape) > 2 else x.shape[1]

        # Reshaping to match expected input shape, e.g., (batch_size, num_sentences, feature_dim)
        x = tf.reshape(x, (batch_size, num_sentences, feature_dim))

        uit = K.tanh(K.dot(x, self.W) + self.b)
        ait = K.dot(uit, self.u)

        ait = K.squeeze(ait, -1)  # Remove the last axis
        ait = K.expand_dims(ait, -1)  # Add an extra dimension for softmax
        ait = tf.nn.softmax(ait, axis=1)  # Apply softmax along the correct axis

        ait = K.expand_dims(ait, axis=-1)  # Add back a dimension for consistency
        weighted_input = x * ait
        return K.sum(weighted_input, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
# from sklearn.model_selection import train_test_split

# # Convert the 'train', 'validation', and 'test' splits to pandas DataFrames
# train_df = dataset_dict['train'].to_pandas()
# val_df = dataset_dict['validation'].to_pandas()
# test_df = dataset_dict['test'].to_pandas()

# # Split the 'train' data into train_data and temp_data (for further validation/test split)
# train_data, temp_data = train_test_split(train_df, test_size=0.2, random_state=42)

# # Further split temp_data into val_data and test_data
# val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# # Check the shapes of the newly split data
# print("New Training data shape:", train_data.shape)
# print("New Validation data shape:", val_data.shape)
# print("New Test data shape:", test_data.shape)


In [ ]:
# from datasets import Dataset

# train_dataset = Dataset.from_pandas(train_data)
# val_dataset = Dataset.from_pandas(val_data)
# test_dataset = Dataset.from_pandas(test_data)

In [ ]:
from datasets import DatasetDict

# Define dataset_dict using DatasetDict
dataset_dict = DatasetDict({
    "train": train_df,
    "validation": val_df,
    "test": test_df
})

# Verify the dataset
print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 10
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 10
    })
})


In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

# Define model name
model_name = "google/pegasus-cnn_dailymail"

# Load the tokenizer and model
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    word_input = Input(shape=(word_count,), dtype='int32')
    word_embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(word_input)
    word_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(word_embedding)
    word_attention = WordAttention()(word_bi_lstm)
    word_encoder = tf.keras.models.Model(inputs=word_input, outputs=word_attention)

    sentence_input = Input(shape=(sentence_count, word_count), dtype='int32')
    sentence_encoder = tf.keras.layers.TimeDistributed(word_encoder)(sentence_input)
    sentence_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(sentence_encoder)
    sentence_attention = SentenceAttention()(sentence_bi_lstm)

    return tf.keras.models.Model(inputs=sentence_input, outputs=sentence_attention)

In [ ]:
import numpy as np
vocab_size = 10000
embedding_dim = 300
sentence_count = 10
word_count = 20
embedding_matrix = np.random.rand(vocab_size, embedding_dim)

hierarchical_model = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)

In [ ]:
import torch

In [ ]:
def preprocess_function(batch, tokenizer, model, vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    batch_size = 4  # Small batch size for Kaggle notebook

    combined_text = [
        f"{article}"
        for article in zip(batch['article'])
    ]

    # Clear CUDA cache before tokenization
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Encode input text
    inputs = tokenizer(
        combined_text,
        truncation=True,
        padding="max_length",
        max_length=min(512, sentence_count * word_count),
        return_tensors="pt"
    )

    # Encode target summaries
    targets = tokenizer(
        batch['highlights'],
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Return only the necessary fields for training
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"]
    }

# Process the dataset
processed_dataset = dataset_dict.map(
    lambda batch: preprocess_function(
        batch, tokenizer, model, vocab_size,
        embedding_dim, sentence_count, word_count,
        embedding_matrix
    ),
    batched=True,
    batch_size=4,
    num_proc=1,
    load_from_cache_file=False
)

In [ ]:
processed_dataset = processed_dataset.remove_columns(dataset_dict["train"].column_names)

In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-finetuned",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    weight_decay=0.001,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    predict_with_generate=True,
    remove_unused_columns=False,
)


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
def evaluate_summaries(dataset, model, tokenizer):
   references = []
   predictions = []

   for sample in dataset:
       # Combine input attributes similar to preprocessing
       input_text = f"{sample['Headline']} {sample['Content']} {sample['Category']}"
       reference_summary = sample['Human Summary']
       references.append(reference_summary)

       # Generate summary
       inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(model.device)
       summary_ids = model.generate(
           inputs["input_ids"],
           max_length=128,
           num_beams=4,
           early_stopping=True
       )
       generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
       predictions.append(generated_summary)

   return references, predictions

In [ ]:
test_references, test_predictions = evaluate_summaries(test_dataset, model, tokenizer)


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

def calculate_average_precision(test_references, test_predictions):
    # Create a set of all unique words across human and predicted summaries
    all_words = set(word for summary in test_references for word in summary.lower().split()) | \
                set(word for summary in test_predictions for word in summary.lower().split())

    # Convert human summaries and predicted summaries into binary vectors
    human_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_references
    ]
    predicted_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_predictions
    ]

    # Calculate the average precision score
    ap_scores = []
    for true, pred in zip(human_vectors, predicted_vectors):
        try:
            ap_scores.append(average_precision_score(true, pred))
        except ValueError:
            ap_scores.append(0.0)

    return np.mean(ap_scores)

In [ ]:
ap_score = calculate_average_precision(test_references, test_predictions)
print(f"Average Precision Score: {ap_score}")

In [ ]:
!pip install evaluate
!pip install rouge_score
import evaluate
rouge = evaluate.load("rouge")

In [ ]:

results = rouge.compute(predictions=test_predictions, references=test_references)

In [ ]:
print(f"ROUGE-1: {results['rouge1']}")
print(f"ROUGE-2: {results['rouge2']}")
print(f"ROUGE-L: {results['rougeL']}")